In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Target Paths
PROCESSED_DIR = "***/data/processed"
FIGURES_DIR = "***/reports/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Plotting Configuration
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

def mann_kendall_test(x, alpha=0.05):
    """Calculates non-parametric Mann-Kendall trend statistic and Sen's slope."""
    n = len(x)
    s = 0
    for k in range(n - 1):
        for j in range(k + 1, n):
            s += np.sign(x[j] - x[k])
            
    unique_x = np.unique(x)
    g = len(unique_x)
    if n == g:
        var_s = (n * (n - 1) * (2 * n + 5)) / 18
    else:
        tp = np.array([len(np.where(x == u)[0]) for u in unique_x])
        var_s = (n * (n - 1) * (2 * n + 5) - np.sum(tp * (tp - 1) * (2 * tp + 5))) / 18
        
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0
        
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    
    slopes = []
    for k in range(n - 1):
        for j in range(k + 1, n):
            slopes.append((x[j] - x[k]) / (j - k))
    sen_slope = np.median(slopes)
    
    return z, p, sen_slope

def run_statistical_analysis():
    global_df = pd.read_csv(os.path.join(PROCESSED_DIR, "climate_energy_global_merged.csv"))
    country_df = pd.read_csv(os.path.join(PROCESSED_DIR, "climate_energy_by_country.csv"))

    print("==================================================")
    print("🔬 RESEARCH STATISTICAL ANALYSIS RESULTS")
    print("==================================================")

    # 1. Non-Parametric Trend Testing (Mann-Kendall)
    z_temp, p_temp, slope_temp = mann_kendall_test(global_df["temp_anomaly_global"].values)
    z_co2, p_co2, slope_co2 = mann_kendall_test(global_df["co2"].values)

    print(f"Global Temp Anomaly Trend  | Z = {z_temp:.3f} | p = {p_temp:.4e} | Sen's Slope = {slope_temp:.4f} °C/yr")
    print(f"Global CO2 Emissions Trend | Z = {z_co2:.3f} | p = {p_co2:.4e} | Sen's Slope = {slope_co2:.2f} Mt/yr")

    # 2. Linear Regression Model
    slope, intercept, r_val, p_val, std_err = stats.linregress(global_df["co2"], global_df["temp_anomaly_global"])
    print(f"\nLinear Regression (CO2 vs Temp Anomaly):")
    print(f"  R² = {r_val**2:.4f}")
    print(f"  Slope = {slope:.6f} °C per million Mt CO2")
    print(f"  p-value = {p_val:.4e}")

    # ==================================================
    # FIGURE GENERATION
    # ==================================================

    # Figure 1: Global Temperature Anomaly Trajectory with Trend
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
    ax.plot(global_df["year"], global_df["temp_anomaly_global"], color="#1f77b4", linewidth=1.5, label="Observed Anomaly (°C)")
    ax.axhline(0, color="black", linestyle="--", alpha=0.5, linewidth=0.8)
    
    global_df["temp_ma10"] = global_df["temp_anomaly_global"].rolling(10).mean()
    ax.plot(global_df["year"], global_df["temp_ma10"], color="#d62728", linewidth=2.2, label="10-Year Moving Average")
    
    ax.set_title("Global Land-Ocean Temperature Index (1880–2024)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Temperature Anomaly (°C relative to 1951-1980)")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#cccccc", framealpha=1.0)
    plt.tight_layout()
    fig1_path = os.path.join(FIGURES_DIR, "fig1_global_temp_anomaly.png")
    plt.savefig(fig1_path)
    plt.close()
    print(f"Saved: {fig1_path}")

    # Figure 2: CO2 Accumulation vs. Global Temperature Correlation
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    sns.regplot(
        data=global_df, x="co2", y="temp_anomaly_global", ax=ax,
        scatter_kws={"alpha":0.6, "color":"#2ca02c"}, line_kws={"color":"#d62728", "linewidth":2}
    )
    ax.set_title(f"Global Temperature Anomaly vs. Cumulative $\\mathrm{{CO_2}}$ Emissions ($R^2 = {r_val**2:.3f}$)", fontweight="bold")
    ax.set_xlabel("Annual Global $\\mathrm{{CO_2}}$ Emissions (Million Metric Tons)")
    ax.set_ylabel("Global Temperature Anomaly (°C)")
    plt.tight_layout()
    fig2_path = os.path.join(FIGURES_DIR, "fig2_co2_temp_correlation.png")
    plt.savefig(fig2_path)
    plt.close()
    print(f"Saved: {fig2_path}")

    # Figure 3: Regional CO2 Trajectories
    fig, ax = plt.subplots(figsize=(9, 5), dpi=300)
    for country in ["United States", "China", "European Union (27)", "India"]:
        c_data = country_df[country_df["country"] == country]
        ax.plot(c_data["year"], c_data["co2"], label=country, linewidth=2)
        
    ax.set_title("Annual $\\mathrm{{CO_2}}$ Emissions Trajectories by Key Economic Regions", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Annual $\\mathrm{{CO_2}}$ Emissions (Million Metric Tons)")
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#cccccc", framealpha=1.0)
    plt.tight_layout()
    fig3_path = os.path.join(FIGURES_DIR, "fig3_regional_co2_trajectories.png")
    plt.savefig(fig3_path)
    plt.close()
    print(f"Saved: {fig3_path}")

    # Figure 4: Rolling 20-Year Correlation Dynamics
    global_df["rolling_corr"] = global_df["co2"].rolling(20).corr(global_df["temp_anomaly_global"])
    fig, ax = plt.subplots(figsize=(9, 4), dpi=300)
    ax.plot(global_df["year"], global_df["rolling_corr"], color="#9467bd", linewidth=2)
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title("20-Year Rolling Pearson Correlation ($\\mathrm{{CO_2}}$ vs. Temperature Anomaly)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Pearson Correlation Coefficient (r)")
    ax.set_ylim(-0.5, 1.05)
    plt.tight_layout()
    fig4_path = os.path.join(FIGURES_DIR, "fig4_rolling_correlation.png")
    plt.savefig(fig4_path)
    plt.close()
    print(f"Saved: {fig4_path}")

    # Figure 5: Multi-Variable Correlation Heatmap
    fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=300)
    corr_vars = ["co2", "temp_anomaly_global", "population", "gdp", "primary_energy_consumption"]
    avail_vars = [v for v in corr_vars if v in global_df.columns]
    
    label_mapping = {
        "co2": "$\mathrm{CO_2}$ Emissions",
        "temp_anomaly_global": "Temperature Anomaly",
        "population": "Population",
        "gdp": "GDP",
        "primary_energy_consumption": "Primary Energy Consumption"
    }
    
    corr_matrix = global_df[avail_vars].corr()
    corr_matrix.rename(index=label_mapping, columns=label_mapping, inplace=True)
    
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", ax=ax, cbar=True)
    ax.set_title("Correlation Heatmap of Key Global Macro-Climate Indicators", fontweight="bold")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig5_path = os.path.join(FIGURES_DIR, "fig5_correlation_matrix.png")
    plt.savefig(fig5_path)
    plt.close()
    print(f"Saved: {fig5_path}")

if __name__ == "__main__":
    run_statistical_analysis()

<>:149: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:149: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/var/folders/h4/34l5wj_13dgg7yf7fpzdm4f80000gn/T/ipykernel_66622/160663938.py:149: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  "co2": "$\mathrm{CO_2}$ Emissions",


🔬 RESEARCH STATISTICAL ANALYSIS RESULTS
Global Temp Anomaly Trend  | Z = 12.921 | p = 0.0000e+00 | Sen's Slope = 0.0080 °C/yr
Global CO2 Emissions Trend | Z = 17.353 | p = 0.0000e+00 | Sen's Slope = 258.27 Mt/yr

Linear Regression (CO2 vs Temp Anomaly):
  R² = 0.8731
  Slope = 0.000031 °C per million Mt CO2
  p-value = 5.7015e-66
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig1_global_temp_anomaly.png
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig2_co2_temp_correlation.png
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig3_regional_co2_trajectories.png
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Tracking/data_analysis_portfolio/project-3/reports/figures/fig4_rolling_correlation.png
Saved: /Users/maharhandi/Desktop/Maha/DATA_ANALYST/Portfolio_Trac